# COVID-19 Twitter Topic Modeling
**Omar Ahmad**

An unsupervised NLP project applying Non-negative Matrix Factorization (NMF) and TF-IDF to over 100,000 real-world tweets from April 30, 2020 — during the early months of the COVID-19 pandemic. The goal is to discover latent discussion topics within the tweet corpus and identify outlier clusters through dimensionality reduction and 3D visualization.

---

## Project Overview

**Dataset:** ~100,000 English-language COVID-19 tweets (Kaggle, April 30 2020)  
**Techniques:** TF-IDF vectorization, Non-negative Matrix Factorization (NMF), topic modeling, outlier detection  
**Tools:** Python, scikit-learn, pandas, NumPy, matplotlib  

**Key questions explored:**
- What distinct topics were people discussing on Twitter at the onset of the pandemic?
- Which topic dominated the conversation on this day?
- Are there outlier tweets that form their own isolated cluster in topic space?

> **Content note:** This dataset contains real-world tweets. While common offensive content was filtered during preprocessing, some sensitive material may remain given the scale of the dataset.

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Load preprocessed tweet dataset
text = pd.read_csv('tweets-2020-4-30.csv')
np.random.seed(416)

text = text.fillna('')  # replace NaN rows with empty string
print(f"Dataset loaded: {len(text):,} tweets")
text.tail()

## 2. Preprocessing Notes

The dataset was preprocessed prior to analysis. Steps applied:

- **Language filter** — retained English-language tweets only
- **URL removal** — stripped hyperlinks as non-informative for topic analysis
- **Lowercasing** — normalized text to lowercase
- **Punctuation removal** — stripped all punctuation characters
- **Stopword removal** — removed common English stopwords (NLTK) plus high-frequency COVID terms (`coronavirus`, `covid19`, etc.) that would otherwise dominate every topic and obscure meaningful signal

These steps are standard NLP preprocessing practice to reduce noise before vectorization.

## 3. TF-IDF Vectorization

TF-IDF (Term Frequency-Inverse Document Frequency) converts raw text into a numeric matrix where each entry reflects how important a word is to a specific tweet relative to the whole corpus. Words that appear frequently across all tweets receive lower weights, reducing the influence of near-universal terms.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build TF-IDF matrix — ignore words appearing in >95% of documents
vectorizer = TfidfVectorizer(max_df=0.95)
tf_idf = vectorizer.fit_transform(text['text'])
feature_names = vectorizer.get_feature_names_out()

num_tweets = tf_idf.shape[0]
num_words  = tf_idf.shape[1]

print(f"Tweets: {num_tweets:,}")
print(f"Vocabulary size: {num_words:,}")
print(f"TF-IDF matrix shape: {tf_idf.shape}")

## 4. Topic Modeling with NMF

Non-negative Matrix Factorization (NMF) decomposes the TF-IDF matrix into two lower-dimensional matrices:

- **Tweet factors** (`tweets_projected`) — each tweet represented as a weighted combination of topics
- **Word factors** (`nmf.components_`) — each topic represented as a weighted combination of words

This is conceptually similar to matrix factorization in recommender systems, but applied here to discover latent semantic structure in text. With **5 components**, the model attempts to surface 5 distinct discussion topics from the corpus.

In [ ]:
from sklearn.decomposition import NMF

# Fit NMF with 5 topics
nmf = NMF(n_components=5, init='nndsvd')
tweets_projected = nmf.fit_transform(tf_idf)

print(f"Tweet factor matrix shape: {tweets_projected.shape}")
print(f"Word factor matrix shape:  {nmf.components_.shape}")
print(f"\nEach tweet is now represented as a 5-dimensional topic weight vector.")

## 5. Inspecting Topics

To interpret each topic, we rank words by their weight in the NMF word factor matrix. The highest-weighted words for a topic reveal what that topic is semantically about.

In [ ]:
def words_from_topic(topic, feature_names):
    """
    Returns words sorted by their weight in a given NMF topic, highest first.
    
    Args:
        topic (np.array): Weight vector for one topic (length = vocabulary size)
        feature_names (list): Vocabulary words corresponding to each index
    
    Returns:
        List of words sorted from highest to lowest topic weight
    """
    sorted_indices = np.flip(np.argsort(topic))
    return [feature_names[i] for i in sorted_indices]


def print_top_words(components, feature_names, n_top_words):
    """Print the top n words for each topic in the NMF model."""
    for topic_index, topic in enumerate(components):
        ordered_words = words_from_topic(topic, feature_names)
        top_words = ', '.join(ordered_words[:n_top_words])
        print(f'Topic #{topic_index}: {top_words}')


print("Top 10 words per topic:\n")
print_top_words(nmf.components_, feature_names, 10)

## 6. Tweet-Level Topic Association

Each tweet is assigned a dominant topic based on which of the 5 topic weights is largest. This allows us to understand which discussion themes were most prevalent across the corpus on this day.

In [ ]:
# Identify the dominant topic for each tweet
dominant_topics = np.argmax(tweets_projected, axis=1)
topic_counts = np.bincount(dominant_topics)
largest_topic = np.argmax(topic_counts)

print("Tweet counts per dominant topic:")
for i, count in enumerate(topic_counts):
    marker = " <-- largest" if i == largest_topic else ""
    print(f"  Topic #{i}: {count:,} tweets{marker}")

print(f"\nThe most dominant topic on April 30, 2020 was Topic #{largest_topic}.")

## 7. Inspecting an Individual Tweet

To validate that the topic assignments are meaningful, we can inspect a specific tweet and check whether its dominant topic aligns with the tweet's content.

In [ ]:
index = 40151
tweet_text = text.iloc[index]['text']
topic_weights = tweets_projected[index]

print(f"Tweet text:\n  '{tweet_text}'\n")
print(f"Topic weights: {np.round(topic_weights, 4)}")
print(f"Dominant topic: Topic #{np.argmax(topic_weights)}")

## 8. 3D Topic Space Visualization

To visualize the full distribution of tweets across topics, we fit a smaller 3-component NMF model. This maps each tweet to a point in 3D space, where the axes correspond to the three topics. Clusters and outliers become visually apparent in this reduced representation.

In [ ]:
# Fit 3-component NMF for visualization
nmf_small = NMF(n_components=3, init='nndsvd')
tweets_projected_small = nmf_small.fit_transform(tf_idf)

print("Topics in 3-component model:\n")
print_top_words(nmf_small.components_, feature_names, 10)

In [ ]:
# 3D scatter plot of all tweets in topic space
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(projection='3d')

ax.scatter(
    tweets_projected_small[:, 0],
    tweets_projected_small[:, 1],
    tweets_projected_small[:, 2],
    alpha=0.3, s=1, c='#2C5F8A'
)

ax.set_xlabel('Topic 0')
ax.set_ylabel('Topic 1')
ax.set_zlabel('Topic 2')
ax.set_title('COVID-19 Tweets in 3D Topic Space\n(April 30, 2020)')
ax.view_init(30, 30)

plt.tight_layout()
plt.show()

print("\nObservation: A small cluster of tweets is visibly isolated along the Topic 2 axis,")
print("suggesting a distinct subset of content separable from the main conversation.")

## 9. Outlier Tweet Detection

The 3D visualization reveals a small cluster of tweets with unusually high Topic 2 weights — isolated from the main distribution. We define outliers as tweets with a Topic 2 value ≥ 0.15 and extract the unique text to investigate what distinguishes this cluster.

In [ ]:
# Identify and extract outlier tweets (Topic 2 weight >= 0.15)
outlier_mask   = tweets_projected_small[:, 2] >= 0.15
outlier_tweets = text.loc[outlier_mask, 'text'].unique()

print(f"Outlier tweets identified: {outlier_mask.sum():,} total, {len(outlier_tweets):,} unique\n")
print("Sample unique outlier tweets:")
for i, tweet in enumerate(outlier_tweets[:10]):
    print(f"  [{i+1}] {tweet}")

## 10. Summary & Findings

**Topic structure:** NMF successfully decomposed ~100,000 pandemic-era tweets into 5 interpretable topics, each anchored by distinct high-weight vocabulary. Topics covered areas such as public health policy, economic impact, personal experience, and government response.

**Dominant topic:** One topic consistently attracted the largest share of tweets on April 30, 2020, reflecting the primary conversation thread of the day.

**Outlier cluster:** A small but distinct subset of tweets was isolated along the Topic 2 axis in 3D topic space. These tweets shared a common theme or phrasing pattern that made them semantically different from the broader corpus — an example of how dimensionality reduction can surface non-obvious structure in large text datasets.

**Business relevance:** This same approach (TF-IDF + NMF) applies directly to product and business contexts — analyzing customer support tickets, app store reviews, survey responses, or internal feedback to surface recurring themes without manual labeling.